In [1]:
from google.colab import files
uploaded=files.upload()


Saving sales.csv to sales.csv


In [6]:
from google.colab import files
uploaded=files.upload()


Saving product.csv to product.csv
Saving sales1.csv to sales1.csv


In [7]:
import pandas as pd

df = pd.read_csv("sales1.csv")

df['revenue'] = df['quantity'] * df['sale_price']
df['cost_total'] = df['quantity'] * df['cost']
df['profit'] = df['revenue'] - df['cost_total']

store_summary = df.groupby('store_id')[['revenue', 'profit']].sum().reset_index()

product_summary = df.groupby('product_id')[['revenue', 'profit']].sum().reset_index()

print(store_summary)
print(product_summary)

store_summary.to_csv("store_summary.csv", index=False)
product_summary.to_csv("product_summary.csv", index=False)

   store_id  revenue  profit
0         1   162600   33200
1         2    29300   12200
   product_id  revenue  profit
0           1    96000   16000
1           2    57000   12000
2           3    12800    4800
3           4     4500    1500
4           5     4800    2400
5           6     8400    3900
6           7     4800    2800
7           8     3600    2000


In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum as _sum

spark = SparkSession.builder.appName("RetailAnalysis").getOrCreate()

df = spark.read.csv("sales.csv", header=True, inferSchema=True)

df = df.withColumn("revenue", df.quantity * df.sale_price)

df = df.withColumn("cost_total", df.quantity * df.cost)

df = df.withColumn("profit", df.revenue - df.cost_total)

store_summary = df.groupBy("store_id") \
    .agg(_sum("revenue").alias("total_revenue"),
         _sum("profit").alias("total_profit"))

underperforming_products = df.groupBy("product_id") \
    .agg(_sum("quantity").alias("total_sold")) \
    .filter("total_sold < 3")

store_summary.show()
underperforming_products.show()

store_summary.write.mode("overwrite").csv("output/store_summary")

+--------+-------------+------------+
|store_id|total_revenue|total_profit|
+--------+-------------+------------+
|       1|       153000|       28000|
|       2|         4500|        1500|
+--------+-------------+------------+

+----------+----------+
|product_id|total_sold|
+----------+----------+
|         1|         2|
+----------+----------+



In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum

spark = SparkSession.builder.appName("RetailAnalysis").getOrCreate()
products_df = spark.read.csv("/content/product.csv", header=True, inferSchema=True)
sales_df = spark.read.csv("/content/sales.csv", header=True, inferSchema=True)
products_df = products_df.withColumnRenamed("cost", "product_cost")
df = sales_df.join(products_df, on="product_id", how="inner")
df = df.withColumn("revenue", col("quantity") * col("sale_price")) \
       .withColumn("cost_total", col("quantity") * col("product_cost")) \
       .withColumn("profit", col("revenue") - col("cost_total"))
category_metrics = df.groupBy("category") \
    .agg(
        sum("revenue").alias("total_revenue"),
        sum("profit").alias("total_profit")
    )

print("Category Metrics:")
category_metrics.show()
category_metrics.write.mode("overwrite").csv("/content/output/category_metrics")

top_products = df.groupBy("product_name") \
    .agg(sum("quantity").alias("total_sold")) \
    .orderBy(col("total_sold").desc())

print("Top Products:")
top_products.show(3)
print("Final Schema:")
df.printSchema()

Category Metrics:
+-----------+-------------+------------+
|   category|total_revenue|total_profit|
+-----------+-------------+------------+
|Electronics|       157500|       22500|
+-----------+-------------+------------+

Top Products:
+------------+----------+
|product_name|total_sold|
+------------+----------+
|  Headphones|         5|
|       Phone|         3|
|      Laptop|         2|
+------------+----------+

Final Schema:
root
 |-- product_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- sale_price: integer (nullable = true)
 |-- cost: integer (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product_cost: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- cost_total: integer (nullable = true)
 |-- profit: integer 